In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)

base = Path.cwd()

if not (base / "data" / "raw").is_dir():
    base = base.parent

raw_dir = base / "data" / "raw"
output_dir = base / "data" / "processed"

if not raw_dir.is_dir():
    raise FileNotFoundError(
        "Start Jupyter from the project root or notebooks folder."
    )

print("Project:", base)
print("Raw data:", raw_dir)

Project: c:\Users\USER\lol_worlds_predictor
Raw data: c:\Users\USER\lol_worlds_predictor\data\raw


In [2]:
files = sorted(raw_dir.glob("*.csv"))

if not files:
    raise FileNotFoundError(f"No CSV files found in {raw_dir}")

print("Files found:", len(files))

for file in files:
    print(file.name)

Files found: 10
2017_LoL_esports_match_data_from_OraclesElixir.csv
2018_LoL_esports_match_data_from_OraclesElixir.csv
2019_LoL_esports_match_data_from_OraclesElixir.csv
2020_LoL_esports_match_data_from_OraclesElixir.csv
2021_LoL_esports_match_data_from_OraclesElixir.csv
2022_LoL_esports_match_data_from_OraclesElixir.csv
2023_LoL_esports_match_data_from_OraclesElixir.csv
2024_LoL_esports_match_data_from_OraclesElixir.csv
2025_LoL_esports_match_data_from_OraclesElixir.csv
2026_LoL_esports_match_data_from_OraclesElixir.csv


In [3]:
required_columns = [
    "gameid",
    "date",
    "league",
    "teamname",
    "position",
    "result",
]

optional_columns = [
    "teamid",
    "side",
    "patch",
    "golddiffat15",
    "dragons",
    "opp_dragons",
    "towers",
    "opp_towers",
    "barons",
    "opp_barons",
]

In [4]:
frames = []
file_audit = []

for file in files:
    header = pd.read_csv(file, nrows=0).columns

    missing_columns = set(required_columns) - set(header)

    if missing_columns:
        raise ValueError(
            f"{file.name} is missing columns: {missing_columns}"
        )

    selected_columns = required_columns + [
        column
        for column in optional_columns
        if column in header
    ]

    frame = pd.read_csv(
        file,
        usecols=selected_columns,
        dtype={
            "gameid": "string",
            "league": "string",
            "teamname": "string",
        },
        low_memory=False,
    )

    team_rows = frame.loc[frame["position"].eq("team")].copy()
    team_rows["source_file"] = file.name

    frames.append(team_rows)

    file_audit.append(
        {
            "source_file": file.name,
            "raw_rows": len(frame),
            "team_rows": len(team_rows),
        }
    )

    print(f"Loaded {file.name}: {len(team_rows):,} team rows")

teams = pd.concat(frames, ignore_index=True)
original_team_rows = len(teams)

display(pd.DataFrame(file_audit))
display(teams.head())

Loaded 2017_LoL_esports_match_data_from_OraclesElixir.csv: 11,008 team rows
Loaded 2018_LoL_esports_match_data_from_OraclesElixir.csv: 13,470 team rows
Loaded 2019_LoL_esports_match_data_from_OraclesElixir.csv: 16,250 team rows
Loaded 2020_LoL_esports_match_data_from_OraclesElixir.csv: 19,492 team rows
Loaded 2021_LoL_esports_match_data_from_OraclesElixir.csv: 24,596 team rows
Loaded 2022_LoL_esports_match_data_from_OraclesElixir.csv: 25,038 team rows
Loaded 2023_LoL_esports_match_data_from_OraclesElixir.csv: 22,196 team rows
Loaded 2024_LoL_esports_match_data_from_OraclesElixir.csv: 20,408 team rows
Loaded 2025_LoL_esports_match_data_from_OraclesElixir.csv: 20,088 team rows
Loaded 2026_LoL_esports_match_data_from_OraclesElixir.csv: 19,636 team rows


,source_file,raw_rows,team_rows
0,2017_LoL_esports_match_data_from_OraclesElixir...,66048,11008
1,2018_LoL_esports_match_data_from_OraclesElixir...,80820,13470
2,2019_LoL_esports_match_data_from_OraclesElixir...,97500,16250
3,2020_LoL_esports_match_data_from_OraclesElixir...,116952,19492
4,2021_LoL_esports_match_data_from_OraclesElixir...,147576,24596
5,2022_LoL_esports_match_data_from_OraclesElixir...,150228,25038
6,2023_LoL_esports_match_data_from_OraclesElixir...,133176,22196
7,2024_LoL_esports_match_data_from_OraclesElixir...,122448,20408
8,2025_LoL_esports_match_data_from_OraclesElixir...,120528,20088
9,2026_LoL_esports_match_data_from_OraclesElixir...,117816,19636


,gameid,league,date,patch,side,position,teamname,teamid,result,dragons,opp_dragons,barons,opp_barons,towers,opp_towers,golddiffat15,source_file
0,TREU/1001490056,EU CS,2017-01-10 15:44:06,6.24,Blue,team,Fnatic Academy,oe:team:47ae4f5f4aea5a7a0ab0b9778844cc2,1,2.0,1.0,1.0,0.0,11.0,4.0,2864.0,2017_LoL_esports_match_data_from_OraclesElixir...
1,TREU/1001490056,EU CS,2017-01-10 15:44:06,6.24,Red,team,EURONICS Gaming,oe:team:514d848d968399a239cdcc3d1412f2d,0,1.0,2.0,0.0,1.0,4.0,11.0,-2864.0,2017_LoL_esports_match_data_from_OraclesElixir...
2,TREU/1001490066,EU CS,2017-01-10 16:21:23,6.24,Blue,team,AlienTech eSports,oe:team:92ebcf3407842256ee7954c5e2390a8,0,0.0,2.0,0.0,1.0,3.0,7.0,-4520.0,2017_LoL_esports_match_data_from_OraclesElixir...
3,TREU/1001490066,EU CS,2017-01-10 16:21:23,6.24,Red,team,Team Larssen,NaN,1,2.0,0.0,1.0,0.0,7.0,3.0,4520.0,2017_LoL_esports_match_data_from_OraclesElixir...
4,TREU/1001490072,EU CS,2017-01-10 17:01:50,6.24,Blue,team,Bobaski,NaN,0,0.0,3.0,0.0,1.0,4.0,9.0,984.0,2017_LoL_esports_match_data_from_OraclesElixir...


In [5]:
for column in ["gameid", "league", "teamname"]:
    teams[column] = (
        teams[column]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

teams["date"] = pd.to_datetime(
    teams["date"],
    utc=True,
    errors="coerce",
    format="mixed",
)

teams["result"] = pd.to_numeric(
    teams["result"],
    errors="coerce",
)

required_team_fields = [
    "gameid",
    "date",
    "league",
    "teamname",
    "result",
]

print("Missing required fields:")
display(teams[required_team_fields].isna().sum())

print("Result values:")
display(teams["result"].value_counts(dropna=False))

print("Date range:", teams["date"].min(), "to", teams["date"].max())

Missing required fields:


gameid      0
date        0
league      0
teamname    1
result      0
dtype: int64

Result values:


result
0    96094
1    96088
Name: count, dtype: int64

Date range: 2017-01-10 15:44:06+00:00 to 2026-10-26 21:54:37+00:00


In [6]:
identity_columns = ["gameid", "date", "league"]

missing_identity = teams[identity_columns].isna().any(axis=1)

unidentified_rows = teams.loc[missing_identity].copy()
identified = teams.loc[~missing_identity].copy()

print("Unidentified rows:", len(unidentified_rows))
print("Identified rows:", len(identified))

display(unidentified_rows.head())

Unidentified rows: 0
Identified rows: 192182


,gameid,league,date,patch,side,position,teamname,teamid,result,dragons,opp_dragons,barons,opp_barons,towers,opp_towers,golddiffat15,source_file


In [7]:
identified["match_id"] = (
    identified["date"].astype("string")
    + "|"
    + identified["league"]
    + "|"
    + identified["gameid"]
)

display(
    identified[
        ["match_id", "gameid", "date", "league", "teamname"]
    ].head()
)

,match_id,gameid,date,league,teamname
0,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,Fnatic Academy
1,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming
2,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports
3,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,Team Larssen
4,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,TREU/1001490072,2017-01-10 17:01:50+00:00,EU CS,Bobaski


In [8]:
comparison_columns = [
    column
    for column in identified.columns
    if column != "source_file"
]

duplicate_mask = identified.duplicated(
    subset=comparison_columns,
    keep="first",
)

duplicate_rows = identified.loc[duplicate_mask].copy()
identified = identified.loc[~duplicate_mask].copy()

print("Exact duplicate rows removed:", len(duplicate_rows))

Exact duplicate rows removed: 0


In [9]:
id_contexts = (
    identified[["gameid", "date", "league"]]
    .drop_duplicates()
    .groupby("gameid")
    .size()
    .rename("context_count")
)

multiple_context_ids = id_contexts.loc[id_contexts.gt(1)]

multiple_context_details = (
    identified.loc[
        identified["gameid"].isin(multiple_context_ids.index),
        ["gameid", "date", "league", "source_file"],
    ]
    .drop_duplicates()
    .sort_values(["gameid", "date", "league"])
)

print("Game IDs with multiple contexts:", len(multiple_context_ids))

display(multiple_context_details.head(20))

Game IDs with multiple contexts: 0


,gameid,date,league,source_file


In [10]:
identified["required_fields_present"] = (
    identified[required_team_fields]
    .notna()
    .all(axis=1)
)

identified["valid_result"] = identified["result"].isin([0, 1])

checks = identified.groupby("match_id").agg(
    row_count=("teamname", "size"),
    unique_teams=("teamname", "nunique"),
    total_wins=("result", "sum"),
    all_fields_present=("required_fields_present", "all"),
    all_results_binary=("valid_result", "all"),
)

checks["bad_row_count"] = checks["row_count"].ne(2)
checks["bad_team_count"] = checks["unique_teams"].ne(2)
checks["bad_winner_count"] = checks["total_wins"].ne(1)
checks["missing_required_fields"] = ~checks["all_fields_present"]
checks["nonbinary_results"] = ~checks["all_results_binary"]

reason_columns = [
    "bad_row_count",
    "bad_team_count",
    "bad_winner_count",
    "missing_required_fields",
    "nonbinary_results",
]

checks["valid"] = ~checks[reason_columns].any(axis=1)

display(checks.head())

,row_count,unique_teams,total_wins,all_fields_present,all_results_binary,bad_row_count,bad_team_count,bad_winner_count,missing_required_fields,nonbinary_results,valid
match_id,,,,,,,,,,,
2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,2,2,1,True,True,False,False,False,False,False,True
2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,2,2,1,True,True,False,False,False,False,False,True
2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,2,2,1,True,True,False,False,False,False,False,True
2017-01-10 17:55:36+00:00|EU CS|TREU/1001490081,2,2,1,True,True,False,False,False,False,False,True
2017-01-10 18:57:51+00:00|EU CS|TREU/1001490084,2,2,1,True,True,False,False,False,False,False,True


In [11]:
valid_ids = checks.index[checks["valid"]]

rejected_games = checks.loc[~checks["valid"]].copy()

rejected_team_rows = identified.loc[
    ~identified["match_id"].isin(valid_ids)
].copy()

clean_teams = identified.loc[
    identified["match_id"].isin(valid_ids)
].copy()

clean_teams["result"] = clean_teams["result"].astype("int64")

print("Accepted games:", len(valid_ids))
print("Rejected game groups:", len(rejected_games))
print("Accepted team records:", len(clean_teams))

print("\nRejection reasons — a game can have multiple reasons:")
display(rejected_games[reason_columns].sum())

display(rejected_games.head())

Accepted games: 96075
Rejected game groups: 16
Accepted team records: 192150

Rejection reasons — a game can have multiple reasons:


bad_row_count               0
bad_team_count             13
bad_winner_count            3
missing_required_fields     1
nonbinary_results           0
dtype: int64

,row_count,unique_teams,total_wins,all_fields_present,all_results_binary,bad_row_count,bad_team_count,bad_winner_count,missing_required_fields,nonbinary_results,valid
match_id,,,,,,,,,,,
2018-09-19 12:27:00+00:00|LDL|4092-5286,2,2,0,True,True,False,False,True,False,False,False
2020-11-18 23:51:32+00:00|UPL|NA1/3669211958,2,1,1,True,True,False,True,False,False,False,False
2020-11-19 00:47:20+00:00|UPL|NA1/3669212337,2,1,1,True,True,False,True,False,False,False,False
2020-11-29 01:41:26+00:00|LAS|ESPORTSTMNT03/1590473,2,1,1,True,True,False,True,False,False,False,False
2020-11-29 02:37:52+00:00|LAS|ESPORTSTMNT03/1600527,2,1,1,True,True,False,True,False,False,False,False


In [12]:
ordered = clean_teams.sort_values(
    ["date", "match_id", "teamname"]
).copy()

ordered["team_index"] = ordered.groupby("match_id").cumcount()

pair_columns = [
    "match_id",
    "date",
    "league",
    "teamname",
    "result",
]

team_a = ordered.loc[
    ordered["team_index"].eq(0),
    pair_columns,
].copy()

team_b = ordered.loc[
    ordered["team_index"].eq(1),
    pair_columns,
].copy()

paired = team_a.merge(
    team_b,
    on="match_id",
    suffixes=("_a", "_b"),
    validate="one_to_one",
)

assert paired["date_a"].eq(paired["date_b"]).all()
assert paired["league_a"].eq(paired["league_b"]).all()
assert paired["teamname_a"].ne(paired["teamname_b"]).all()
assert paired["result_a"].add(paired["result_b"]).eq(1).all()

display(paired.head())

,match_id,date_a,league_a,teamname_a,result_a,date_b,league_b,teamname_b,result_b
0,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,0,2017-01-10 15:44:06+00:00,EU CS,Fnatic Academy,1
1,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,0,2017-01-10 16:21:23+00:00,EU CS,Team Larssen,1
2,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,2017-01-10 17:01:50+00:00,EU CS,Bobaski,0,2017-01-10 17:01:50+00:00,EU CS,Nerv,1
3,2017-01-10 17:55:36+00:00|EU CS|TREU/1001490081,2017-01-10 17:55:36+00:00,EU CS,Team Forge,1,2017-01-10 17:55:36+00:00,EU CS,Tricked Esport,0
4,2017-01-10 18:57:51+00:00|EU CS|TREU/1001490084,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,0,2017-01-10 18:57:51+00:00,EU CS,LDLC OL,1


In [15]:
matches = (
    paired.rename(
        columns={
            "date_a": "date",
            "league_a": "league",
            "teamname_a": "team_a",
            "teamname_b": "team_b",
        }
    )[
        [
            "match_id",
            "date",
            "league",
            "team_a",
            "team_b",
            "result_a",
        ]
    ]
    .sort_values(["date", "match_id"])
    .reset_index(drop=True)
)

if matches.empty:
    raise ValueError("No valid matches remain after validation.")

assert matches["match_id"].is_unique
assert matches.notna().all().all()
assert matches["date"].is_monotonic_increasing
assert len(matches) == len(valid_ids)
assert len(clean_teams) == 2 * len(matches)

print("Match table shape:", matches.shape)
display(matches.head(10))

Match table shape: (96075, 6)


,match_id,date,league,team_a,team_b,result_a
0,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,0
1,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,0
2,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,0
3,2017-01-10 17:55:36+00:00|EU CS|TREU/1001490081,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,1
4,2017-01-10 18:57:51+00:00|EU CS|TREU/1001490084,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,0
5,2017-01-11 15:33:10+00:00|EU CS|TREU/1001490139,2017-01-11 15:33:10+00:00,EU CS,Fnatic Academy,LDLC OL,1
6,2017-01-11 15:46:32+00:00|EU CS|TREU/1001490141,2017-01-11 15:46:32+00:00,EU CS,ALTERNATE aTTaX,AlienTech eSports,0
7,2017-01-11 16:44:20+00:00|EU CS|TREU/1001490143,2017-01-11 16:44:20+00:00,EU CS,ASUS ROG Army,Nerv,0
8,2017-01-11 17:53:03+00:00|EU CS|TREU/1001490147,2017-01-11 17:53:03+00:00,EU CS,Tricked Esport,devils.one inStreamly,0
9,2017-01-11 18:59:33+00:00|EU CS|TREU/1001490149,2017-01-11 18:59:33+00:00,EU CS,Bobaski,EURONICS Gaming,0


In [16]:
team_statistics = (
    clean_teams.drop(
        columns=["required_fields_present", "valid_result"]
    )
    .sort_values(["date", "match_id", "teamname"])
    .reset_index(drop=True)
)

print("Team-statistics table shape:", team_statistics.shape)

available_statistics = [
    column
    for column in optional_columns
    if column in team_statistics.columns
]

print("Missing optional statistics:")
display(team_statistics[available_statistics].isna().sum())

display(team_statistics.head())

Team-statistics table shape: (192150, 18)
Missing optional statistics:


teamid           2584
side                0
patch             448
golddiffat15    19366
dragons             0
opp_dragons         0
towers              0
opp_towers          0
barons            198
opp_barons        198
dtype: int64

,gameid,league,date,patch,side,position,teamname,teamid,result,dragons,opp_dragons,barons,opp_barons,towers,opp_towers,golddiffat15,source_file,match_id
0,TREU/1001490056,EU CS,2017-01-10 15:44:06+00:00,6.24,Red,team,EURONICS Gaming,oe:team:514d848d968399a239cdcc3d1412f2d,0,1.0,2.0,0.0,1.0,4.0,11.0,-2864.0,2017_LoL_esports_match_data_from_OraclesElixir...,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056
1,TREU/1001490056,EU CS,2017-01-10 15:44:06+00:00,6.24,Blue,team,Fnatic Academy,oe:team:47ae4f5f4aea5a7a0ab0b9778844cc2,1,2.0,1.0,1.0,0.0,11.0,4.0,2864.0,2017_LoL_esports_match_data_from_OraclesElixir...,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056
2,TREU/1001490066,EU CS,2017-01-10 16:21:23+00:00,6.24,Blue,team,AlienTech eSports,oe:team:92ebcf3407842256ee7954c5e2390a8,0,0.0,2.0,0.0,1.0,3.0,7.0,-4520.0,2017_LoL_esports_match_data_from_OraclesElixir...,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066
3,TREU/1001490066,EU CS,2017-01-10 16:21:23+00:00,6.24,Red,team,Team Larssen,NaN,1,2.0,0.0,1.0,0.0,7.0,3.0,4520.0,2017_LoL_esports_match_data_from_OraclesElixir...,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066
4,TREU/1001490072,EU CS,2017-01-10 17:01:50+00:00,6.24,Blue,team,Bobaski,NaN,0,0.0,3.0,0.0,1.0,4.0,9.0,984.0,2017_LoL_esports_match_data_from_OraclesElixir...,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072


In [17]:
summary = pd.Series(
    {
        "source_files": len(files),
        "original_team_rows": original_team_rows,
        "unidentified_rows": len(unidentified_rows),
        "exact_duplicate_rows_removed": len(duplicate_rows),
        "rejected_game_groups": len(rejected_games),
        "accepted_games": len(matches),
        "game_ids_with_multiple_contexts": len(multiple_context_ids),
        "earliest_game": matches["date"].min(),
        "latest_game": matches["date"].max(),
    },
    name="value",
)

display(summary)

print("\nAccepted games by year:")
display(
    matches.groupby(matches["date"].dt.year)
    .size()
    .rename("games")
)

print("\nAccepted games by competition:")
display(matches["league"].value_counts())

source_files                                              10
original_team_rows                                    192182
unidentified_rows                                          0
exact_duplicate_rows_removed                               0
rejected_game_groups                                      16
accepted_games                                         96075
game_ids_with_multiple_contexts                            0
earliest_game                      2017-01-10 15:44:06+00:00
latest_game                        2026-10-26 21:54:37+00:00
Name: value, dtype: object


Accepted games by year:


date
2017     5504
2018     6734
2019     8125
2020     9740
2021    12291
2022    12519
2023    11097
2024    10203
2025    10044
2026     9818
Name: games, dtype: int64


Accepted games by competition:


league
LDL     7019
LPL     6918
LCK     4978
LCKC    2664
VCS     2526
        ... 
MSC       25
Riot      20
AC        19
LGL        8
WLRQ       6
Name: count, Length: 124, dtype: Int64

In [18]:
output_dir.mkdir(parents=True, exist_ok=True)

matches.to_csv(
    output_dir / "matches_clean.csv",
    index=False,
)

team_statistics.to_csv(
    output_dir / "team_game_statistics_clean.csv",
    index=False,
)

rejected_games.to_csv(
    output_dir / "rejected_games.csv",
    index=True,
)

rejected_team_rows.to_csv(
    output_dir / "rejected_team_rows.csv",
    index=False,
)

unidentified_rows.to_csv(
    output_dir / "unidentified_team_rows.csv",
    index=False,
)

duplicate_rows.to_csv(
    output_dir / "duplicate_team_rows.csv",
    index=False,
)

multiple_context_details.to_csv(
    output_dir / "game_ids_multiple_contexts.csv",
    index=False,
)

pd.DataFrame(file_audit).to_csv(
    output_dir / "input_file_audit.csv",
    index=False,
)

summary.to_csv(
    output_dir / "cleaning_summary.csv",
    header=True,
)

print("Saved outputs to:", output_dir)

Saved outputs to: c:\Users\USER\lol_worlds_predictor\data\processed
